# Access the analysis ready ORAS5 data

This notebook provides an example of how to open the reanalysis-oras5 Zarr store using xarray.
You will need to insert your CDS API key where indicated in the following code cell (available on [your profile page](https://cds-dev-cci2.copernicus-climate.eu/profile)).
For more information on using the Data Store Analysis Ready Datasets, please see the [user documentation pages](https://cds-dev-cci2.copernicus-climate.eu/datasets/how-to-use-the-dss-arco-dataset).

In [1]:
import os
cdsapi_key = "<INSERT-CDS-API-KEY-HERE>"

# The following attempts to find the CDSAPI key in your environment.
if cdsapi_key == "<INSERT-CDS-API-KEY-HERE>":
    cdsapi_key = None
if cdsapi_key is None:
    cdsapi_key = os.getenv("CDSAPI_KEY")
if cdsapi_key is None and os.path.exists(os.path.expanduser("~/.cdsapirc")):
    with open(os.path.expanduser("~/.cdsapirc"), "r") as f:
        for line in f:
            if line.startswith("key:"):
                cdsapi_key = line.split(":")[1].strip()
                break
if cdsapi_key is None:
    raise ValueError("CDSAPI key not found. Please set the CDSAPI_KEY environment variable or create a ~/.cdsapirc file.")


## Plug and play access

The code below provides a simple plug and play example of how to use the Zarr Store for light-weight access.

In [2]:
import xarray as xr
from obstore.store import HTTPStore
from zarr.storage import ObjectStore

# Time-chunked store is optimised for spatial subsetting across a region.
# obstore uses bundled CA roots (rustls), avoiding macOS SSL cert issues.
# The OPERATIONAL stream holds recent years (~2015+); the CONSOLIDATED stream
# holds the long historical record. Pick the stream that covers your year range.
timechunked_consolidated_url = "https://arco.datastores.ecmwf.int/cadl-arco-time-001/arco/reanalysis_oras5/consolidated/timeChunked.zarr"
timechunked_operational_url = "https://arco.datastores.ecmwf.int/cadl-arco-time-001/arco/reanalysis_oras5/operational/timeChunked.zarr"


def open_stream(url):
    http_store = HTTPStore(
        url,
        client_options={"default_headers": {"Authorization": f"Bearer {cdsapi_key}"}},
    )
    store = ObjectStore(http_store, read_only=True)
    return xr.open_zarr(store)


# Open both streams; the regrid loop below picks the right one per year range.
ds_consolidated = open_stream(timechunked_consolidated_url)
ds_operational = open_stream(timechunked_operational_url)

# Backwards-compatible default handle
ds = ds_consolidated

# Inspect the variables and coordinates
ds


<xarray.Dataset> Size: 253GB
Dimensions:    (time: 684, elevation: 75, latitude: 721, longitude: 1440)
Coordinates:
  * time       (time) datetime64[ns] 5kB 1958-01-01 1958-02-01 ... 2014-12-01
  * elevation  (elevation) float32 300B -5.902e+03 -5.698e+03 ... -1.556 -0.5058
  * latitude   (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * longitude  (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
Data variables: (12/15)
    vosaline   (time, elevation, latitude, longitude) float32 213GB ...
    iicethic   (time, latitude, longitude) float32 3GB ...
    iicevelu   (time, latitude, longitude) float32 3GB ...
    iicevelv   (time, latitude, longitude) float32 3GB ...
    ileadfra   (time, latitude, longitude) float32 3GB ...
    so20chgt   (time, latitude, longitude) float32 3GB ...
    ...         ...
    somxl010   (time, latitude, longitude) float32 3GB ...
    somxl030   (time, latitude, longitude) float32 3GB ...
    sosaline   (time, latitude, longitude) float32 3GB ...
    sossheig   (time, latitude, longitude) float32 3GB ...
    sosstsst   (time, latitude, longitude) float32 3GB ...
    sozotaux   (time, latitude, longitude) float32 3GB ...

## Subset to the ERA5 region & grid, then export CSV + metadata

Select the three variables over the year range, subset to the same El Nino Pacific region as the ERA5 dataset, and interpolate onto the same 2-degree grid.

To avoid interpolation artifacts along coasts/islands (linear interpolation propagates land `NaN`s into neighbouring ocean cells), we **fill land gaps in the source grid by nearest-ocean extrapolation before interpolating**, then **re-apply the ERA5 land–sea mask** so the final ORAS5 ocean mask matches ERA5 exactly. Finally we write a CSV plus a metadata JSON.


In [3]:
import os
import json
import numpy as np
import pandas as pd
from scipy import ndimage

DATA_DIR = '../../data/ORAS5'
os.makedirs(DATA_DIR, exist_ok=True)

# ERA5 land-check file used to build the shared land-sea mask on the 2-degree grid
ERA5_LANDCHECK = '../../data/ERA5/era5_combined_1980_2026_landcheck.csv'

# Requested variables (ORAS5 / NEMO short names)
VARS = [
    'so20chgt',   # depth_of_20_c_isotherm
    'sohtc300',   # ocean_heat_content_for_the_upper_300m
    'sossheig',   # sea_surface_height
]

# Year ranges to (re)generate. Blocks >= 2015 come from the operational stream.
YEAR_RANGES = [
    (1980, 1993),
    (1994, 2004),
    (2005, 2014),
    (2015, 2026),
]

# Same 2-degree grid as ERA5 (area [30, 120, -30, -80], grid [2.0, 2.0])
target_lat = np.arange(-30, 30 + 2, 2.0)
target_lon = np.arange(120, 280 + 2, 2.0)

# Shared ERA5 ocean mask on the 2-degree target grid (lsm < 0.5 == ocean)
edf = pd.read_csv(ERA5_LANDCHECK, skipinitialspace=True)
edf.columns = [c.strip() for c in edf.columns]
lsm_da = edf.groupby(['latitude', 'longitude'])['lsm'].mean().to_xarray()
ocean_mask = lsm_da.reindex(latitude=target_lat, longitude=target_lon) < 0.5


def _fill_nearest_2d(a):
    """Fill NaNs in a 2D array with the value of the nearest non-NaN cell.

    Uses a Euclidean distance transform to find, for every masked cell, the
    index of the closest valid cell. This extrapolates ocean values over land
    so that the subsequent linear interpolation is not contaminated by land NaNs.
    """
    mask = np.isnan(a)
    if not mask.any() or mask.all():
        return a
    idx = ndimage.distance_transform_edt(mask, return_distances=False, return_indices=True)
    return a[tuple(idx)]


def regrid_range(start_year, end_year):
    # Recent years live in the operational stream; historical in consolidated.
    source = ds_operational if start_year >= 2015 else ds_consolidated

    sub = source[VARS].sel(time=slice(f'{start_year}', f'{end_year}'))

    # El Nino Pacific box. Native lon is -180..180 and crosses the dateline,
    # so convert to 0-360 first.
    sub = sub.assign_coords(longitude=(sub.longitude % 360)).sortby('longitude')
    sub = sub.sel(latitude=slice(-30, 30), longitude=slice(120, 280))

    # Fill land gaps on the SOURCE grid before interp so linear interpolation
    # is not contaminated by land NaNs along coasts/islands.
    sub_filled = xr.apply_ufunc(
        _fill_nearest_2d,
        sub,
        input_core_dims=[['latitude', 'longitude']],
        output_core_dims=[['latitude', 'longitude']],
        vectorize=True,
        dask='parallelized' if sub.chunks else None,
        output_dtypes=[np.float32],
    )

    oras = sub_filled.interp(latitude=target_lat, longitude=target_lon)

    # Re-apply the ERA5 land-sea mask so ORAS5 ocean cells match ERA5 exactly.
    oras = oras.where(ocean_mask)

    # Trigger the remote reads and bring the subset into memory
    oras = oras.load()

    # Save metadata
    metadata = {
        'dimensions': dict(oras.sizes),
        'variables': {},
        'global_attributes': dict(oras.attrs),
    }
    for var in oras.variables:
        metadata['variables'][var] = {
            'dims': list(oras[var].dims),
            'shape': list(oras[var].shape),
            'dtype': str(oras[var].dtype),
            'attributes': dict(oras[var].attrs),
        }
    meta_path = os.path.join(DATA_DIR, f'oras5_{start_year}_{end_year}_metadata.json')
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)

    # Convert to DataFrame and save as CSV
    csv_path = os.path.join(DATA_DIR, f'oras5_{start_year}_{end_year}.csv')
    df = oras.to_dataframe().reset_index()
    df.to_csv(csv_path, index=False)
    print(f"  {start_year}-{end_year}: {len(df)} rows -> {csv_path}")
    return csv_path


for start_year, end_year in YEAR_RANGES:
    print(f"Regridding {start_year}-{end_year} ...")
    regrid_range(start_year, end_year)
print("Done.")


Regridding 1980-1993 ...
  1980-1993: 421848 rows -> ../../data/ORAS5/oras5_1980_1993.csv
Regridding 1994-2004 ...
  1994-2004: 331452 rows -> ../../data/ORAS5/oras5_1994_2004.csv
Regridding 2005-2014 ...
  2005-2014: 301320 rows -> ../../data/ORAS5/oras5_2005_2014.csv
Regridding 2015-2026 ...
  2015-2026: 344007 rows -> ../../data/ORAS5/oras5_2015_2026.csv
Done.


In [6]:
import os
import glob
import pandas as pd

DATA_DIR = '../../data/ORAS5'

# Combine the per-range CSVs into a single 1980-2026 CSV + Parquet
range_files = [
    os.path.join(DATA_DIR, f'oras5_{s}_{e}.csv') for s, e in YEAR_RANGES
]
frames = [pd.read_csv(f) for f in range_files]
combined = pd.concat(frames, ignore_index=True)
combined = combined.sort_values(['time', 'latitude', 'longitude']).reset_index(drop=True)

combined_csv = os.path.join(DATA_DIR, 'oras5_combined_1980_2026.csv')
combined_parquet = os.path.join(DATA_DIR, 'oras5_combined_1980_2026.parquet')
combined.to_csv(combined_csv, index=False)
combined.to_parquet(combined_parquet, index=False, engine='fastparquet')

print(f"Combined rows: {len(combined)}")
print(f"CSV     -> {combined_csv}")
print(f"Parquet -> {combined_parquet}")
combined[['so20chgt', 'sohtc300', 'sossheig']].isna().sum()


Combined rows: 1398627
CSV     -> ../../data/ORAS5/oras5_combined_1980_2026.csv
Parquet -> ../../data/ORAS5/oras5_combined_1980_2026.parquet


so20chgt    122540
sohtc300    122540
sossheig    122540
dtype: int64

## Upload to Google Drive

Create an `ORAS` folder under the shared Drive parent and upload only the two derived files: the CSV and the metadata JSON (not the raw data).


In [13]:
import os
import pickle
import mimetypes
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

SCOPES = ['https://www.googleapis.com/auth/drive']
TOKEN_PATH = '../../token.pickle'
CLIENT_SECRETS = '../../client_secrets.json'

# Parent Drive folder under which all data folders are uploaded
DRIVE_PARENT_ID = '1zLJgkYkrM1LRgtl7v5sfAQ4aits9zfUq'

DATA_DIR = '../../data/ORAS5'

# Year range (must match the download cell above)
START_YEAR = 1980
END_YEAR = 1993

# Only upload the CSV and the metadata (two files)
FILES_TO_UPLOAD = [
    os.path.join(DATA_DIR, f'oras5_{START_YEAR}_{END_YEAR}.csv'),
    os.path.join(DATA_DIR, f'oras5_{START_YEAR}_{END_YEAR}_metadata.json'),
]


def get_drive_service():
    creds = None
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, 'rb') as f:
            creds = pickle.load(f)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRETS, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_PATH, 'wb') as f:
            pickle.dump(creds, f)
    return build('drive', 'v3', credentials=creds)


def get_or_create_folder(service, name, parent_id):
    """Return the Drive folder id for `name` under `parent_id`, creating it if needed."""
    query = (
        f"name = '{name}' and mimeType = 'application/vnd.google-apps.folder' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    response = service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
    files = response.get('files', [])
    if files:
        return files[0]['id']
    folder = service.files().create(
        body={'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]},
        fields='id'
    ).execute()
    return folder['id']


service = get_drive_service()

# Create (or reuse) the ORAS folder to hold the CSV and metadata
oras_folder_id = get_or_create_folder(service, 'ORAS', DRIVE_PARENT_ID)
print(f"Folder: ORAS (Drive id: {oras_folder_id})")

for path in FILES_TO_UPLOAD:
    name = os.path.basename(path)
    mimetype = mimetypes.guess_type(path)[0] or 'application/octet-stream'
    media = MediaFileUpload(path, mimetype=mimetype, resumable=True)
    result = service.files().create(
        body={'name': name, 'parents': [oras_folder_id]},
        media_body=media,
        fields='id, name'
    ).execute()
    print(f"  Uploaded: {result['name']} (id: {result['id']})")

print("Done.")


Folder: ORAS (Drive id: 155COy8_s9dO2WXtDAF1R8_A0yahmL8VO)
  Uploaded: oras5_1980_1993.csv (id: 1jToczw7Z1qBcH21gxH2B2pGDaaJ1EX5j)
  Uploaded: oras5_1980_1993_metadata.json (id: 17lFLDa1MlKgkm6tGFbuxkpDjkpqFe89G)
Done.


In [10]:
import os
import numpy as np
import pandas as pd

# Verify the ORAS5 output matches the ERA5 grid & region
oras_csv = '../../data/ORAS5/oras5_2024.csv'
era5_csv = '../../data/ERA5/era5_singlelevels_1980.csv'

o = pd.read_csv(oras_csv)
print("ORAS5 columns:", list(o.columns))
print("rows:", len(o))

olat = np.sort(o['latitude'].unique())
olon = np.sort(o['longitude'].unique())
otime = sorted(o['time'].unique())
print(f"latitudes : {olat.min()}..{olat.max()} step {np.unique(np.diff(olat))} ({olat.size} pts)")
print(f"longitudes: {olon.min()}..{olon.max()} step {np.unique(np.diff(olon))} ({olon.size} pts)")
print(f"time steps: {len(otime)} ({otime[0]} .. {otime[-1]})")

# Cross-check the grid against ERA5 (longitudes compared in 0-360)
if os.path.exists(era5_csv):
    e = pd.read_csv(era5_csv)
    elat = np.sort(e['latitude'].unique())
    elon = np.sort((e['longitude'].unique() % 360))
    print("\nGrid matches ERA5:")
    print("  latitudes match :", np.allclose(olat, elat))
    print("  longitudes match:", np.allclose(olon, elon))

print("\nExpected rows = 31 lat x 81 lon x 12 months =", 31 * 81 * 12)


ORAS5 columns: ['time      ', ' latitude', ' longitude', ' so20chgt              ', ' sohtc300     ', ' sossheig']
rows: 30132


KeyError: 'latitude'

This mirrors the ERA5 workflow: same region, same 2-degree grid, exported to CSV + metadata, with only those two files uploaded to the `ORAS` Drive folder.
